<a href="https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Rule in plain words:** Prioritize a page for a human refresh review when it has not been updated for at least 180 days and it still has at least 500 search impressions in the trailing 90-day window. Rank qualifying pages by impressions so that the most visible pages are reviewed first.

**Why these signals:** Days since last update is a transparent staleness signal linked to refresh-priority logic. Impressions are a transparent visibility/volume signal: a refresh review may be more useful when a page still receives search exposure. These signals justify a review queue; they do **not** prove that a refresh will improve performance.

**One rule reason code:** `stale_but_visible` — both thresholds are met. Pages that do not meet both thresholds receive `not_selected`; this is an explanation for exclusion, not a second scoring rule.

The decline label is used only to audit signal patterns in Section 1 and is never used to calculate the score or select pages. The rule uses no future-window fields, `trend_pct`, `trend_direction`, `is_declining_label`, or product-generated flags.


In [1]:
# Load the starter CSV, verify required columns, and inspect both signals against the observed decline label.
# The label is used for signal auditing only; it is NOT used in the rule score.

from pathlib import Path
import subprocess
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the user's repository when running from a local clone/Colab; otherwise clone it.
cwd = Path.cwd()
repo_candidates = [cwd, cwd / "flyrank-ml-internship"]
repo_root = next((p for p in repo_candidates if (p / "work" / "notebooks").exists()), None)

if repo_root is None:
    repo_root = cwd / "flyrank-ml-internship"
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Chaman4211/flyrank-ml-internship.git", str(repo_root)],
            check=True,
        )

# The teaching CSV is in the starter repository; never download or print client-identifying data.
possible_csvs = [
    repo_root / "data" / "raw" / "content_refresh_anonymized.csv",
    cwd / "data" / "raw" / "content_refresh_anonymized.csv",
    cwd / "flyrank-ml-internship-starter" / "data" / "raw" / "content_refresh_anonymized.csv",
]
csv_path = next((p for p in possible_csvs if p.exists()), None)

if csv_path is None:
    starter_dir = cwd / "flyrank-ml-internship-starter"
    if not starter_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Chaman4211/flyrank-ml-internship-starter.git", str(starter_dir)],
            check=True,
        )
    candidate = starter_dir / "data" / "raw" / "content_refresh_anonymized.csv"
    if candidate.exists():
        csv_path = candidate

if csv_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Make sure the FlyRank starter repository is accessible and its data file is present."
    )

df = pd.read_csv(csv_path)
required = ["content_id", "client_id", "days_since_last_update", "impressions_90d", "trend_direction"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Required columns missing from starter CSV: {missing}")

# Numeric conversion; preserve missing values as missing for signal tables.
for col in ["days_since_last_update", "impressions_90d"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# This is the documented observed label. It is for descriptive signal checks only.
df["is_declining_label"] = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)

print(f"Loaded starter data: {len(df):,} rows × {df.shape[1]:,} columns")
print(f"Data source: {csv_path.name} (path shown without exposing any client information)")
print(f"Observed decline-label base rate for signal auditing only: {df['is_declining_label'].mean():.3f}")

# Signal 1: update age (staleness), a refresh-priority signal.
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=["0–30 days", "31–90 days", "91–180 days", "181+ days"],
    include_lowest=True,
)
staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .reset_index()
)
staleness_table["decline_rate_pct"] = (100 * staleness_table["decline_rate"]).round(2)
print("\nSignal check 1 — days since last update (larger bucket = staler)")
display(staleness_table[["staleness_bucket", "n", "decline_rate_pct"]])

def verdict_from_rates(table, bucket_col, rate_col, expected_direction):
    usable = table.dropna(subset=[rate_col]).copy()
    usable = usable[usable["n"] > 0]
    if len(usable) < 3 or usable[rate_col].nunique() < 2:
        return "MIXED", "Too few distinct bucket rates to establish a clear pattern."
    x = np.arange(len(usable), dtype=float)
    rho = pd.Series(x).corr(usable[rate_col].reset_index(drop=True), method="spearman")
    if pd.isna(rho) or abs(rho) < 0.10:
        return "FALSE", f"No meaningful monotonic pattern in these buckets (Spearman trend={rho:.2f})."
    if expected_direction * rho >= 0.20:
        return "CONFIRMED", f"The bucket pattern broadly matches the expected direction (Spearman trend={rho:.2f})."
    if expected_direction * rho <= -0.20:
        return "OPPOSITE", f"The observed pattern runs against the expected direction (Spearman trend={rho:.2f})."
    return "MIXED", f"The pattern is weak or inconsistent (Spearman trend={rho:.2f})."

staleness_verdict, staleness_explanation = verdict_from_rates(
    staleness_table, "staleness_bucket", "decline_rate", expected_direction=1
)
print(f"Verdict: {staleness_verdict} — {staleness_explanation}")

# Signal 2: search impressions (visibility/volume), a quick-win/impact-priority signal.
df["impression_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 99, 499, 2999, np.inf],
    labels=["0–99", "100–499", "500–2,999", "3,000+"],
    include_lowest=True,
)
impression_table = (
    df.groupby("impression_bucket", observed=False)
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .reset_index()
)
impression_table["decline_rate_pct"] = (100 * impression_table["decline_rate"]).round(2)
print("\nSignal check 2 — search impressions in the trailing 90 days")
display(impression_table[["impression_bucket", "n", "decline_rate_pct"]])

# For the volume signal, this checks whether higher-volume buckets are more/less associated
# with the observed decline label. This is descriptive and does not establish causal impact.
impression_verdict, impression_explanation = verdict_from_rates(
    impression_table, "impression_bucket", "decline_rate", expected_direction=-1
)
print(f"Verdict: {impression_verdict} — {impression_explanation}")

signal_results = {
    "staleness": {"verdict": staleness_verdict, "explanation": staleness_explanation},
    "impressions": {"verdict": impression_verdict, "explanation": impression_explanation},
}


Loaded starter data: 30,000 rows × 45 columns
Data source: content_refresh_anonymized.csv (path shown without exposing any client information)
Observed decline-label base rate for signal auditing only: 0.542

Signal check 1 — days since last update (larger bucket = staler)


,staleness_bucket,n,decline_rate_pct
0,0–30 days,20480,51.14
1,31–90 days,175,58.86
2,91–180 days,9171,61.11
3,181+ days,174,47.13


Verdict: MIXED — The pattern is weak or inconsistent (Spearman trend=-0.20).

Signal check 2 — search impressions in the trailing 90 days


,impression_bucket,n,decline_rate_pct
0,0–99,7994,38.90
1,100–499,5280,60.44
2,"500–2,999",8443,62.06
3,"3,000+",8283,57.00


Verdict: OPPOSITE — The observed pattern runs against the expected direction (Spearman trend=0.40).


## 2. Build the ranked queue (writes the CSV)

The rule is fixed before modeling: `days_since_last_update >= 180` **and** `impressions_90d >= 500`. Qualifying pages receive a score equal to their observed trailing-90-day impressions and are sorted from highest to lowest. This is a prioritization heuristic, not a prediction that a refresh will succeed.

The notebook writes `work/outputs/baseline_action_score.csv`. The CSV is a generated data artifact and should remain out of Git; the notebook regenerates it when run. A small metrics JSON is saved separately as the run receipt.


In [2]:
# Apply one transparent rule, rank the full queue, and save the required CSV.
# No label, trend field, future-window value, or product flag enters this score.

stale_cutoff_days = 180
visibility_cutoff_impressions = 500

df["meets_staleness_rule"] = df["days_since_last_update"].ge(stale_cutoff_days).fillna(False)
df["meets_visibility_rule"] = df["impressions_90d"].ge(visibility_cutoff_impressions).fillna(False)
df["selected_for_review"] = df["meets_staleness_rule"] & df["meets_visibility_rule"]

df["score"] = np.where(df["selected_for_review"], df["impressions_90d"], 0.0)
df["reason_code"] = np.where(df["selected_for_review"], "stale_but_visible", "not_selected")
df["action_label"] = np.where(
    df["selected_for_review"],
    "Review for content refresh",
    "No immediate refresh flag",
)

# Rank candidates first; unselected rows follow with score 0. Pseudonymous IDs are not model features.
df["rank"] = (
    df.sort_values(
        ["score", "impressions_90d", "days_since_last_update"],
        ascending=[False, False, False],
        kind="mergesort",
    ).reset_index().index + 1
)
queue = df.sort_values(
    ["score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False],
    kind="mergesort",
).copy()
queue["rank"] = np.arange(1, len(queue) + 1)

# Keep the output limited to pseudonymous IDs and decision-support fields.
output_columns = [
    "rank", "content_id", "client_id", "score", "reason_code", "action_label",
    "days_since_last_update", "impressions_90d",
]
queue_output = queue[output_columns].copy()

outputs_dir = repo_root / "work" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
csv_output_path = outputs_dir / "baseline_action_score.csv"
queue_output.to_csv(csv_output_path, index=False)

candidate_count = int(df["selected_for_review"].sum())
metrics = {
    "assignment": "ML-07 baseline action score",
    "rule": "days_since_last_update >= 180 AND impressions_90d >= 500",
    "score": "impressions_90d for qualifying pages; 0 otherwise",
    "reason_code_for_selected": "stale_but_visible",
    "rows_in_input": int(len(df)),
    "selected_for_review": candidate_count,
    "selected_rate": float(df["selected_for_review"].mean()) if len(df) else None,
    "staleness_cutoff_days": stale_cutoff_days,
    "visibility_cutoff_impressions": visibility_cutoff_impressions,
    "label_used_in_score": False,
    "future_window_features_used": False,
    "output_csv": "work/outputs/baseline_action_score.csv",
    "signal_verdicts": signal_results,
}
metrics_path = outputs_dir / "baseline_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Rows in queue: {len(queue_output):,}")
print(f"Pages selected for human review: {candidate_count:,}")
print(f"Selected rate: {df['selected_for_review'].mean():.2%}")
print(f"CSV written to: {csv_output_path}")
print(f"Metrics JSON written to: {metrics_path}")
print("\nTop 10 rows (pseudonymous IDs only):")
display(queue_output.head(10))


Rows in queue: 30,000
Pages selected for human review: 17
Selected rate: 0.06%
CSV written to: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Metrics JSON written to: /content/flyrank-ml-internship/work/outputs/baseline_metrics.json

Top 10 rows (pseudonymous IDs only):


,rank,content_id,client_id,score,reason_code,action_label,days_since_last_update,impressions_90d
16751,1,content_cf56e2e2e282,client_7f2253d7e2,61678.0,stale_but_visible,Review for content refresh,194,61678
16514,2,content_7368877ea310,client_7f2253d7e2,59472.0,stale_but_visible,Review for content refresh,194,59472
7021,3,content_1bfaa38ff26c,client_7f2253d7e2,25715.0,stale_but_visible,Review for content refresh,194,25715
21268,4,content_0a91db491d14,client_7f2253d7e2,13299.0,stale_but_visible,Review for content refresh,193,13299
11489,5,content_5feee3994adb,client_7f2253d7e2,7812.0,stale_but_visible,Review for content refresh,194,7812
12045,6,content_c2d929d83eaa,client_7f2253d7e2,7558.0,stale_but_visible,Review for content refresh,193,7558
698,7,content_b16bd7307b39,client_7f2253d7e2,4590.0,stale_but_visible,Review for content refresh,194,4590
5327,8,content_fe16a55cd13d,client_7f2253d7e2,4556.0,stale_but_visible,Review for content refresh,194,4556
26810,9,content_ecb6215e79fd,client_7f2253d7e2,4429.0,stale_but_visible,Review for content refresh,194,4429
20837,10,content_928af3e22c80,client_7f2253d7e2,1697.0,stale_but_visible,Review for content refresh,193,1697


## 3. Top-20 review

The task card asks for ten reviewed rows. The code below reviews the top ten selected pages and writes one line per page with the action, reason, confidence note, and a specific way the recommendation could be wrong. Confidence refers to confidence that the **rule conditions were met**, not confidence that a refresh will improve outcomes.


In [3]:
# Review the top 10 selected pages with a skeptic's eye.
selected_queue = queue.loc[queue["selected_for_review"]].head(10).copy()

def review_confidence(row):
    # Rule-confidence only: how far values are above the chosen thresholds.
    age_margin = float(row["days_since_last_update"]) - stale_cutoff_days
    volume_margin = float(row["impressions_90d"]) - visibility_cutoff_impressions
    if age_margin >= 90 and volume_margin >= 2500:
        return "High rule-fit; still needs editorial review."
    if age_margin >= 30 and volume_margin >= 500:
        return "Moderate rule-fit; confirm page context."
    return "Threshold-edge pick; verify before spending time."

def what_could_make_it_wrong(row):
    if row["days_since_last_update"] >= 365:
        return "The page may be evergreen and intentionally unchanged; age alone may not mean it needs a refresh."
    if row["impressions_90d"] < 1000:
        return "The page only narrowly clears the visibility threshold; its business value may be low."
    return "Impressions do not show intent, conversion value, seasonality, or whether the page is already accurate."

review = pd.DataFrame({
    "rank": selected_queue["rank"].astype(int),
    "item_id": selected_queue["content_id"].astype(str),
    "action": selected_queue["action_label"],
    "why_here": selected_queue.apply(
        lambda r: f"Reason code {r['reason_code']}: last update was {int(r['days_since_last_update'])} days ago and trailing-90-day impressions were {int(r['impressions_90d']):,}.",
        axis=1,
    ),
    "confidence_note": selected_queue.apply(review_confidence, axis=1),
    "what_would_make_it_wrong": selected_queue.apply(what_could_make_it_wrong, axis=1),
})

if len(review) == 0:
    print("No rows met both rule thresholds. That is a valid result; do not invent review rows.")
else:
    for _, row in review.iterrows():
        print(
            f"Rank {row['rank']} | {row['item_id']} — Action: {row['action']}. "
            f"Why: {row['why_here']} Confidence: {row['confidence_note']} "
            f"What could make it wrong: {row['what_would_make_it_wrong']}"
        )

print(f"\nTop-10 review lines produced: {len(review)} (target: 10 where at least 10 candidates exist).")


Rank 1 | content_cf56e2e2e282 — Action: Review for content refresh. Why: Reason code stale_but_visible: last update was 194 days ago and trailing-90-day impressions were 61,678. Confidence: Threshold-edge pick; verify before spending time. What could make it wrong: Impressions do not show intent, conversion value, seasonality, or whether the page is already accurate.
Rank 2 | content_7368877ea310 — Action: Review for content refresh. Why: Reason code stale_but_visible: last update was 194 days ago and trailing-90-day impressions were 59,472. Confidence: Threshold-edge pick; verify before spending time. What could make it wrong: Impressions do not show intent, conversion value, seasonality, or whether the page is already accurate.
Rank 3 | content_1bfaa38ff26c — Action: Review for content refresh. Why: Reason code stale_but_visible: last update was 194 days ago and trailing-90-day impressions were 25,715. Confidence: Threshold-edge pick; verify before spending time. What could make it w

## 4. Weak picks + leakage check

Review the lowest-ranked selected pages as deliberately weak picks: they barely pass one or both thresholds and show where the simple rule may waste editorial time. The checks below confirm that the score is not built from the decline label, trend fields, future-window fields, or product-generated flags. If fewer than ten candidates qualify, the notebook reports the actual count rather than fabricating rows.


In [4]:
# Show weak/edge picks, then audit the rule inputs for leakage.

weak_picks = queue.loc[queue["selected_for_review"]].tail(5).sort_values("rank", ascending=True)
if weak_picks.empty:
    print("No qualifying pages: no weak picks can be reviewed without inventing candidates.")
else:
    print("Weakest selected picks (lowest scores among pages that still pass the rule):")
    weak_review = weak_picks[[
        "rank", "content_id", "days_since_last_update", "impressions_90d", "reason_code", "action_label"
    ]].copy()
    weak_review["why_weak"] = weak_review.apply(
        lambda r: (
            "Near the 500-impression cutoff; verify the page has meaningful value."
            if r["impressions_90d"] < 1000
            else "Passes both thresholds but is low-ranked relative to higher-volume pages."
        ),
        axis=1,
    )
    display(weak_review)

# Explicit leakage audit: these fields may exist in the dataset but are never scoring inputs.
forbidden_score_inputs = {
    "trend_pct", "trend_direction", "is_declining_label",
    "future_impressions", "future_clicks", "future_sessions",
    "health_score", "needs_ctr_fix", "is_quick_win",
}
actual_score_inputs = {"days_since_last_update", "impressions_90d"}
used_forbidden = actual_score_inputs.intersection(forbidden_score_inputs)
assert not used_forbidden, f"Forbidden inputs accidentally used in score: {used_forbidden}"
assert "score" in df.columns and df["score"].notna().all()
assert df["score"].eq(0).equals(~df["selected_for_review"]) or (
    (df.loc[~df["selected_for_review"], "score"] == 0).all()
)
print("\nLeakage check passed:")
print("- Score inputs are only days_since_last_update and impressions_90d.")
print("- Decline label/trend fields were used only for descriptive signal tables.")
print("- No future-window or product-generated flag was used to score or rank pages.")
print("- Human review is required; this queue does not claim refreshes cause improvement.")


Weakest selected picks (lowest scores among pages that still pass the rule):


,rank,content_id,days_since_last_update,impressions_90d,reason_code,action_label,why_weak
26840,13,content_7f116ae1f6f5,301,954,stale_but_visible,Review for content refresh,Near the 500-impression cutoff; verify the pag...
26799,14,content_77d4d5930e5e,194,828,stale_but_visible,Review for content refresh,Near the 500-impression cutoff; verify the pag...
7452,15,content_72496874f806,301,821,stale_but_visible,Review for content refresh,Near the 500-impression cutoff; verify the pag...
11630,16,content_6226ee6adc91,183,545,stale_but_visible,Review for content refresh,Near the 500-impression cutoff; verify the pag...
3507,17,content_074ba6ead17b,183,533,stale_but_visible,Review for content refresh,Near the 500-impression cutoff; verify the pag...



Leakage check passed:
- Score inputs are only days_since_last_update and impressions_90d.
- Decline label/trend fields were used only for descriptive signal tables.
- No future-window or product-generated flag was used to score or rank pages.
- Human review is required; this queue does not claim refreshes cause improvement.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.